### Dependencies

In [38]:
# %pip install pandas numpy seaborn matplotlib scikit-learn wfdb 

### Imports

In [39]:
import numpy as np
import pandas as pd
import wfdb

import seaborn as sns
import matplotlib.pyplot as plt

import os

### Loading in Data

In [40]:
VFDB_DIR = "data/mit-bih-malignant-ventricular-ectopy-database-1.0.0"
CUDB_DIR = "data/cu-ventricular-tachyarrhythmia-database-1.0.0"

In [41]:
vfdb_records = sorted(set(f.split(".")[0] for f in os.listdir(VFDB_DIR) if f.endswith(".hea")))
cudb_records = sorted(set(f.split(".")[0] for f in os.listdir(CUDB_DIR) if f.endswith(".hea")))

In [42]:
record_path = f"{VFDB_DIR}/{vfdb_records[0]}"
ann = wfdb.rdann(record_path, "atr")

print(ann.sample[:5])
print(ann.aux_note[:5])

[    18  99624 101499 133092 134038]
['(N\x00', '(VFL\x00', '(N\x00', '(VFL\x00', '(N\x00']


In [43]:
raw = ann.aux_note[0]
cleaned = raw.strip("(\x00")
print(repr(raw), "->", repr(cleaned))

'(N\x00' -> 'N'


In [44]:
fs = wfdb.rdheader(record_path).fs
print(fs)

250


In [45]:
def get_episode_durations(record_path, positive_labels=("VF", "VFL")):
    ann = wfdb.rdann(record_path, "atr")
    fs = wfdb.rdheader(record_path).fs
    
    durations = []
    
    for i in range(len(ann.sample)):
        label = ann.aux_note[i].strip("(\x00")
        
        if label in positive_labels:
            if i + 1 < len(ann.sample):
                start = ann.sample[i]
                end = ann.sample[i + 1]
                duration_sec = (end - start) / fs
                durations.append(duration_sec)
    
    return durations

In [46]:
vfdb_test = get_episode_durations(f"{VFDB_DIR}/418")
print(vfdb_test)

cudb_test = get_episode_durations(f"{CUDB_DIR}/cu01")
print(cudb_test)

[np.float64(7.5), np.float64(3.784), np.float64(3.412), np.float64(4.232), np.float64(5.396), np.float64(4.296), np.float64(2.46), np.float64(2.476), np.float64(2.46), np.float64(6.436), np.float64(4.676), np.float64(3.552), np.float64(4.652), np.float64(1.692), np.float64(9.0), np.float64(16.232), np.float64(2.808), np.float64(8.232), np.float64(5.656), np.float64(4.156), np.float64(4.54), np.float64(3.46), np.float64(2.924), np.float64(9.384), np.float64(3.08), np.float64(6.768), np.float64(1.08), np.float64(1.308), np.float64(8.692), np.float64(7.232), np.float64(7.308), np.float64(2.384), np.float64(0.744), np.float64(1.232), np.float64(1.9), np.float64(3.972), np.float64(1.46), np.float64(5.536), np.float64(1.388), np.float64(9.256), np.float64(0.972), np.float64(4.348), np.float64(6.232), np.float64(1.536), np.float64(1.54), np.float64(1.0), np.float64(1.616), np.float64(1.308), np.float64(3.076), np.float64(1.616), np.float64(1.54), np.float64(2.512), np.float64(1.616), np.float

In [47]:
record_path = f"{VFDB_DIR}/430"
ann = wfdb.rdann(record_path, "atr")

for s, note in zip(ann.sample, ann.aux_note):
    print(s, repr(note))

28 '(BI\x00'
51903 '(VT\x00'
53211 '(BI\x00'
85615 '(NOISE\x00'
93134 '(BI\x00'
94423 '(VT\x00'
94903 '(BI\x00'
98596 '(BI\x00'
99557 '(VFL\x00'
118057 '(NOISE\x00'
120807 '(VF\x00'
131153 '(ASYS\x00'
134519 '(SBR\x00'
138076 '(HGEA\x00'
156576 '(VT\x00'
157923 '(HGEA\x00'
163192 '(VT\x00'
169179 '(ASYS\x00'
170237 '(HGEA\x00'
172275 '(VT\x00'
173891 '(HGEA\x00'
175429 '(VF\x00'
193891 '(ASYS\x00'
196217 '(VF\x00'
273294 '(ASYS\x00'
274391 '(VF\x00'
310544 '(ASYS\x00'
315333 '(SBR\x00'
319871 '(VF\x00'
406794 '(ASYS\x00'
407102 '(VER\x00'
415025 '(VT\x00'
489621 '(VER\x00'
493794 '(VF\x00'


In [48]:
all_durations_with_record = []

for rec in vfdb_records:
    record_path = f"{VFDB_DIR}/{rec}"
    for d in get_episode_durations(record_path):
        all_durations_with_record.append((d, rec))

for rec in cudb_records:
    record_path = f"{CUDB_DIR}/{rec}"
    for d in get_episode_durations(record_path):
        all_durations_with_record.append((d, rec))

In [51]:
suspicious = [(d, rec) for d, rec in all_durations_with_record if d > 100]
print(suspicious)

[(np.float64(520.348), '426'), (np.float64(308.308), '430'), (np.float64(144.612), '430'), (np.float64(347.692), '430')]


In [ ]:
def inspect_long_episodes(record_path, positive_labels=("VF", "VFL"), threshold=100):
    ann = wfdb.rdann(record_path, "atr")
    fs = wfdb.rdheader(record_path).fs
    
    results = []
    for i in range(len(ann.sample)):
        label = ann.aux_note[i].strip("(\x00")
        if label in positive_labels and i + 1 < len(ann.sample):
            duration_sec = (ann.sample[i+1] - ann.sample[i]) / fs
            if duration_sec > threshold:
                next_label = ann.aux_note[i+1].strip("(\x00")
                results.append((label, duration_sec, "->", next_label))
    return results

print(inspect_long_episodes(f"{VFDB_DIR}/426"))
print(inspect_long_episodes(f"{VFDB_DIR}/430"))

[('VF', np.float64(520.348), '->', 'NOISE')]
[('VF', np.float64(308.308), '->', 'ASYS'), ('VF', np.float64(144.612), '->', 'ASYS'), ('VF', np.float64(347.692), '->', 'ASYS')]


In [ ]:
record_path = f"{VFDB_DIR}/426"
ann = wfdb.rdann(record_path, "atr")

for s, note in zip(ann.sample, ann.aux_note):
    print(s, repr(note))

18 '(N\x00'
161432 '(VF\x00'
291519 '(NOISE\x00'
343980 '(VF\x00'
360230 '(NOISE\x00'
395288 '(N\x00'
416961 '(VT\x00'
418384 '(NOISE\x00'
435211 '(VT\x00'
442538 '(SVTA\x00'
444269 '(VT\x00'
448115 '(SVTA\x00'
458522 '(VT\x00'
459791 '(SVTA\x00'
468003 '(VF\x00'
483753 '(NOISE\x00'
499753 '(N\x00'
